# Week 5 – Werkcollege 8: resultaten, peer review en aftrap case

Vandaag lever je niets nieuws in. Je ziet hoe Bot v3 presteerde, je test live, je beoordeelt kaarten en grafieken van klasgenoten, en de docent trapt de laatste groepscase af.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Live testen | 15 min | Bot v3 tegen klasgenoot testen |
| Resultaten: v3 vs v1 | 20 min | Toernooi ophalen, hele reis v1 -> v3 vergelijken |
| Peer review via de API | 25 min | 3 inzendingen beoordelen (matplotlib, plotly óf folium) |
| Kaart met eindresultaten | 15 min | Dezelfde kaart, nu met echte kleuren |
| Aftrap groepscase | 15 min | Uitleg door docent |


## Deel 1 — Live testen (15 min)

Zoek een klasgenoot op. Wissel `mijn_bot_week5.py` uit en laat beide bots dezelfde combinaties zien, inclusief een paar met een hoge `bluf_kans`.


In [ ]:
import sys
sys.path.append('.')

from mijn_bot_week5 import kies_actie as bot_jij
# from bot_klasgenoot_week5 import kies_actie as bot_klasgenoot

test_gevallen = [
    (["7", "2"], 1000, "tight", 0.1),
    (["7", "2"], 1000, "tight", 0.8),
    (["A", "A"], 1000, "loose", 0.3),
]
for hand, stack, strategie, bluf_kans in test_gevallen:
    print(hand, stack, strategie, bluf_kans, "->", bot_jij(hand, stack, strategie, bluf_kans))
    # print(hand, stack, strategie, bluf_kans, "->", bot_klasgenoot(hand, stack, strategie, bluf_kans))


## Deel 2 — Resultaten: de hele reis van v1 naar v3 (20 min)

De docent heeft het toernooi van deze week vers gedraaid, vergeleken met Week 1 — net als in Werkcollege 5. Haal het op.


In [ ]:
import requests
import pandas as pd

API_URL = "http://localhost:8000"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

response = requests.get(
    f"{API_URL}/toernooi/5",
    params={"student_id": STUDENT_ID, "vergelijk_met_week": 1},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
resultaat = response.json()
resultaat["eindstand_per_bot"]


Bouw dezelfde long-naar-wide tabel als in Werkcollege 5, nu voor Week 1 vs Week 5.


In [ ]:
rijen = []
for bot_naam, eindstand in resultaat["eindstand_per_bot"].items():
    student_id, week_label = bot_naam.split("__w")
    rijen.append({"student_id": student_id, "week": int(week_label), "eindstand": eindstand})

lang_formaat = pd.DataFrame(rijen)
breed_formaat = lang_formaat.pivot(index="student_id", columns="week", values="eindstand")
breed_formaat.columns = ["eindstand_week1", "eindstand_week5"]
breed_formaat["verschil"] = breed_formaat["eindstand_week5"] - breed_formaat["eindstand_week1"]
breed_formaat.sort_values("verschil", ascending=False)


💡 Bonus (optioneel): wil je ook zien hoe je Week 5-bot het deed tegenover je Week 3-bot specifiek? Herhaal Deel 2 met `vergelijk_met_week=3` in plaats van `1`.


## Deel 3 — Peer review via de API (25 min)

Deze week loopt alles door elkaar: matplotlib, Plotly én Folium. Je toon-code moet nu alle drie kunnen afhandelen.


In [ ]:
response = requests.get(
    f"{API_URL}/gallery/5",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
gallery = response.json()
gallery


In [ ]:
import base64
from IPython.display import Image, HTML, display
import plotly.graph_objects as go

eerste = gallery[0]
print(eerste["chart"]["titel"], "|", eerste["chart"]["library"])

if eerste["chart"]["library"] == "matplotlib":
    png_bytes = base64.b64decode(eerste["chart"]["figuur_json"])
    display(Image(png_bytes))
elif eerste["chart"]["library"] == "plotly":
    fig = go.Figure(eerste["chart"]["figuur_json"])
    fig.show()
elif eerste["chart"]["library"] == "folium":
    display(HTML(eerste["chart"]["figuur_json"]))


Beoordeel op dezelfde 3 criteria als altijd — een kaart heeft geen assen, maar heeft wel degelijk een focal point, een kleurkeuze en een titel die iets kan beweren.


In [ ]:
review = {
    "week": 5,
    "anon_id": eerste["anon_id"],
    "focal_point_score": 4,       # pas aan
    "kleur_contrast_score": 3,    # pas aan
    "actietitel_score": 5,        # pas aan
    "opmerking": "optioneel commentaar",
}

response = requests.post(
    f"{API_URL}/peer-review/{STUDENT_ID}",
    json=review,
    headers={"Authorization": f"Bearer {TOKEN}"},
)
response.json()


Herhaal voor de overige 2 inzendingen uit `gallery`.


## Deel 4 — Kaart met eindresultaten (15 min)

Haal `/locaties/5` opnieuw op. Nu het toernooi gedraaid is, is `eindstand` overal gevuld — je kaart uit Werkcollege 7 kleurt zichzelf in.


In [ ]:
import folium

response = requests.get(
    f"{API_URL}/locaties/5",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
locaties = response.json()

kaart_klas = folium.Map(location=[52.1, 5.1], zoom_start=7)
for bot in locaties:
    kleur = "gray" if bot["eindstand"] is None else ("green" if bot["eindstand"] >= 1000 else "red")
    folium.CircleMarker(
        location=[bot["lat"], bot["lon"]],
        radius=8, color=kleur, fill=True, fill_color=kleur,
        popup=f"{bot['student_id']} ({bot['plaatsnaam']}) — eindstand: {bot['eindstand']}",
        tooltip=bot["student_id"],
    ).add_to(kaart_klas)
kaart_klas


🤔 Vergelijk deze kaart met die uit Werkcollege 7 (toen was alles grijs). Verandert dat iets aan hoe je naar dezelfde posities kijkt?


---

## Reflectievragen

🤔 Je hebt nu 3 keer dezelfde workflow doorlopen (Week 1, 3, 5): bot bouwen, testen, inleveren, resultaten ophalen, peer review. Wat is er onderweg makkelijker geworden, puur door herhaling?

🎨 Was een kaart moeilijker te beoordelen op "focal point" dan een lijngrafiek? Waarom wel of niet?

💡 Kijk naar `breed_formaat` uit Deel 2. Is de bot die het meest verbeterde ook de bot die nu bovenaan staat? Zijn dat twee verschillende vragen?

🤔 Deel 4's kaart gebruikt maar 2 kleuren (groen/rood) plus grijs voor onbekend. Wat zou je toevoegen als je ook wilde laten zien hóeveel iemand won, niet alleen of.

---

## Deel 5 — Aftrap groepscase (15 min)

De docent introduceert de derde en laatste groepscase.
